# Deploying a Trained LLM with Gradio — Reference Notebook

> **Reference notebook.** See [`tiny_addition_llm_training.ipynb`](./tiny_addition_llm_training.ipynb)
> for training and saving the model this notebook loads, and [`deployment.md`](./deployment.md) for the
> broader deployment concepts this notebook is a minimal, local instance of.

**Methods covered:**
- Reloading a saved model with `AutoModelForCausalLM.from_pretrained`
- A hand-written autoregressive inference function matched to a fixed-vocabulary custom tokenizer
- Wrapping inference in a web UI with `gradio.Interface` (`fn`, `inputs`, `outputs`, `examples`) and
  launching it

**Use this as a reference when:** you need the smallest possible example of taking a trained model from
"saved on disk" to "reachable through a browser UI" — no server framework, no auth, no scaling.

**Don't use this as a reference for:** production deployment (see [`deployment.md`](./deployment.md) for
load balancing, monitoring, security, and the rest of what a real deployment needs beyond this).


In [ ]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Loads the weights saved by tiny_addition_llm_training.ipynb -- run that notebook first if this
# directory doesn't exist yet.
model = AutoModelForCausalLM.from_pretrained("./tiny_addition_llm/")
model.eval()


In [ ]:
class AdditionTokenizer:
    """Same fixed 13-token vocabulary used at training time -- redefined here (not imported) so this
    notebook stays runnable on its own, independent of the training notebook's process/state."""

    def __init__(self):
        vocab = ["+", "=", "-1", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
        self.pad_token = "-1"
        self.encoder = {str(v): i for i, v in enumerate(vocab)}
        self.decoder = {i: str(v) for i, v in enumerate(vocab)}
        self.pad_token_id = self.encoder[self.pad_token]

    def decode(self, token_ids):
        return " ".join(self.decoder[t] for t in token_ids)

    def __call__(self, text):
        return [self.encoder[t] for t in text.split()]


tokenizer = AdditionTokenizer()
vocab_size = 13


In [ ]:
def predict(text, solution_length=2, model=model):
    # Hand-written token-by-token decoding, matching the training notebook -- this custom, tiny-vocab
    # model doesn't fit the standard `.generate()` utilities the way a normal pretrained checkpoint does.
    model.eval()
    input_ids = torch.tensor(tokenizer(text)).unsqueeze(0)

    solution = []
    for _ in range(solution_length):
        logits = model(input_ids).logits[0, -1]
        predicted = logits[:vocab_size].argmax()
        input_ids = torch.cat((input_ids, predicted.unsqueeze(0).unsqueeze(0)), dim=1)
        solution.append(predicted.cpu().item())

    return tokenizer.decode(solution)


# The model was trained on inputs shaped exactly like "d + d =" (trailing "=", single space between
# tokens) -- deviating from that format is the fastest way to get a nonsense prediction.
print(predict("3 + 5 ="))
print(predict("8 + 1 ="))


In [ ]:
def predict_for_ui(text):
    return predict(text, solution_length=2)


# gr.Interface wires a plain Python function up to a browser UI without any HTML/JS:
#   fn          -- the function called on every submit; its signature drives the input widgets.
#   inputs      -- one gr.Textbox per `fn` argument (here just one: the raw arithmetic expression).
#   outputs     -- one gr.Textbox for `fn`'s return value.
#   title/description -- static text shown above the widgets.
#   examples    -- clickable example inputs, prefilled into the input widget for a quick first try.
webapp = gr.Interface(
    fn=predict_for_ui,
    inputs=[gr.Textbox(
        label="Input",
        lines=1,
        info="Must look like '1 + 2 =', with a single space between each character and single-digit numbers only.",
    )],
    outputs=[gr.Textbox(label="Prediction", lines=1)],
    title="Tiny Addition LLM",
    description="Enter an expression and click Submit to get the model's prediction.",
    examples=["5 + 3 =", "2 + 9 ="],
)


In [ ]:
# share=False (unlike the course original's share=True) keeps this on localhost only. share=True
# additionally tunnels a temporary public URL through Gradio's own servers -- convenient for a quick
# demo, but it exposes this app to the internet, so it's worth choosing deliberately rather than by
# default.
webapp.launch(share=False)
